In [1]:
# Import required libraries
import os
from dotenv import load_dotenv
from litellm import completion
import json
import pandas as pd
import subprocess
import matplotlib.pyplot as plt
import platform
import shutil
from datetime import datetime

# Load environment variables
load_dotenv()

print("Dependencies loaded successfully!")

Dependencies loaded successfully!


In [15]:
class LLMTester:
    def __init__(self):
        # Create output directory if it doesn't exist
        self.output_dir = "llm_test_results"
        os.makedirs(self.output_dir, exist_ok=True)
        
        # Initialize model configurations
        self.models = {
            "anthropic": "claude-3-5-sonnet-20240620",
            "openai": "gpt-4o"
        }
        
        # Generate unique filenames for this test run
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.config_file = os.path.join(self.output_dir, f"promptfoo_config_{timestamp}.json")
        self.test_cases_file = os.path.join(self.output_dir, f"test_cases_{timestamp}.json")
        self.results_file = os.path.join(self.output_dir, f"results_{timestamp}.json")
        
        # Initialize by checking dependencies
        self._check_dependencies()

    def _check_dependencies(self):
        """Check if required dependencies are installed"""
        try:
            # Check Node.js
            node_version = subprocess.run(['node', '--version'], 
                                       capture_output=True, 
                                       text=True)
            print(f"Found Node.js: {node_version.stdout.strip()}")
            
            # Check npm
            npm_version = subprocess.run(['npm', '--version'], 
                                      capture_output=True, 
                                      text=True)
            print(f"Found npm: {npm_version.stdout.strip()}")
            
            # Check PromptFoo
            promptfoo_path = self._get_promptfoo_path()
            if promptfoo_path:
                print(f"Found PromptFoo at: {promptfoo_path}")
            else:
                print("Installing PromptFoo...")
                subprocess.run(['npm', 'install', -g, 'promptfoo'], 
                             check=True)
                
        except Exception as e:
            raise EnvironmentError(f"Dependency check failed: {str(e)}")

    def _get_promptfoo_path(self):
        """Get the full path to the promptfoo executable"""
        cmd = 'promptfoo.cmd' if platform.system() == 'Windows' else 'promptfoo'
        return shutil.which(cmd)

    def create_promptfoo_config(self):
        """
        Creates a PromptFoo configuration that follows their exact schema requirements.
        This implementation uses the most basic supported configuration structure
        to ensure compatibility with PromptFoo's validation system.
        """
        config = {
            "providers": ["openai:chat:gpt-4o-mini" , "anthropic:messages:claude-3-5-sonnet-20241022"],
            "prompts": "Role: You are a customer service representative.\nTask: Help resolve the following customer issue:\nCustomer message: {{customer_message}}\n\nInstructions:\n- Respond professionally and empathetically\n- Address the specific issue mentioned\n- Provide clear next steps\n- Include relevant tracking or order information"
        }
        
        # Let's add detailed logging to help us understand what's being created
        try:
            with open(self.config_file, 'w') as f:
                json.dump(config, f, indent=2)
            print(f"Created PromptFoo configuration at: {self.config_file}")
            
            # Log the exact configuration for verification
            print("\nConfiguration Details:")
            print("Provider Information:")
            for provider in config['providers']:
                print(f"- Provider: {provider['id']}")
                # Safely print part of the credential for verification
                cred = provider['credential']
                if cred:
                    print(f"  Credential: {cred[:8]}... (truncated)")
                else:
                    print("  Warning: No credential found")
                    
            print("\nPrompt Information:")
            for prompt in config['prompts']:
                print(f"- Prompt ID: {prompt['id']}")
                print(f"  Length: {len(prompt['text'])} characters")
            
            return config
            
        except Exception as e:
            print(f"Error creating config file: {str(e)}")
            return None

    def create_test_cases(self):
        """
        Creates test cases using PromptFoo's basic assertion capabilities.
        These test cases focus on fundamental checks that we know are supported.
        """
        test_cases = {
            "tests": [
                {
                    "description": "Basic Customer Service Test",
                    "vars": {
                        "customer_message": "My order #12345 hasn't arrived yet."
                    },
                    "assert": {
                        "contains": "order"
                    }
                }
            ]
        }
        
        try:
            with open(self.test_cases_file, 'w') as f:
                json.dump(test_cases, f, indent=2)
            print(f"\nCreated test cases at: {self.test_cases_file}")
            
            # Log test case details for verification
            print("\nTest Case Details:")
            for idx, test in enumerate(test_cases['tests'], 1):
                print(f"\nTest {idx}:")
                print(f"Description: {test['description']}")
                print(f"Variables: {test['vars']}")
                print(f"Assertions: {test['assert']}")
            
            return test_cases
            
        except Exception as e:
            print(f"Error creating test cases: {str(e)}")
            return None

    def run_evaluation(self):
        """Runs PromptFoo evaluation with enhanced error handling"""
        try:
            # Verify config files exist
            if not os.path.exists(self.config_file):
                print("Configuration file missing. Creating now...")
                self.create_promptfoo_config()
            
            if not os.path.exists(self.test_cases_file):
                print("Test cases file missing. Creating now...")
                self.create_test_cases()
    
            promptfoo_path = self._get_promptfoo_path()
            if not promptfoo_path:
                raise EnvironmentError("PromptFoo not found in system PATH")
    
            print(f"\nRunning evaluation...")
            print(f"Config file: {self.config_file}")
            print(f"Test cases: {self.test_cases_file}")
            print(f"Results will be saved to: {self.results_file}")
    
            # Run PromptFoo with explicit paths
            result = subprocess.run(
                [promptfoo_path, 'eval',
                 '--config', self.config_file,
                 '--tests', self.test_cases_file,
                 '--output', self.results_file],
                capture_output=True,
                text=True
            )
    
            # Check if the command was successful
            if result.returncode != 0:
                print("PromptFoo execution failed:")
                print("Standard output:", result.stdout)
                print("Standard error:", result.stderr)
                return None
    
            # Verify results file was created
            if not os.path.exists(self.results_file):
                print("Warning: Results file was not created")
                print("PromptFoo output:", result.stdout)
                print("PromptFoo errors:", result.stderr)
                return None
    
            # Load and return results
            with open(self.results_file, 'r') as f:
                results = json.load(f)
            
            print("Evaluation completed successfully")
            return pd.DataFrame(results['results'])
    
        except Exception as e:
            print(f"Error during evaluation: {str(e)}")
            return None

    def analyze_results(self, df_results):
        """Analyzes evaluation results with enhanced error handling"""
        if df_results is None or df_results.empty:
            print("No results to analyze")
            return
    
        try:
            print("\n=== Evaluation Summary ===")
            print(f"Total tests run: {len(df_results)}")
            
            if 'pass' in df_results.columns:
                pass_rate = (df_results['pass'].sum() / len(df_results)) * 100
                print(f"Pass rate: {pass_rate:.2f}%")
            
            # Save detailed results
            output_file = os.path.join(self.output_dir, 'detailed_results.csv')
            df_results.to_csv(output_file, index=False)
            print(f"\nDetailed results saved to: {output_file}")
    
        except Exception as e:
            print(f"Error analyzing results: {str(e)}")

In [16]:
# Start fresh with a clean directory
if os.path.exists('llm_test_results'):
    shutil.rmtree('llm_test_results')
os.makedirs('llm_test_results')

# Initialize tester and run tests
tester = LLMTester()
print("Starting new test run with simplified configuration...")

# Create and verify configuration
config = tester.create_promptfoo_config()
if not config:
    print("Failed to create configuration")
    exit(1)

# Create and verify test cases
test_cases = tester.create_test_cases()
if not test_cases:
    print("Failed to create test cases")
    exit(1)

# Run evaluation
results = tester.run_evaluation()
if results is not None:
    tester.analyze_results(results)

Found Node.js: v23.3.0
Found npm: 10.9.0
Found PromptFoo at: /opt/homebrew/bin/promptfoo
Starting new test run with simplified configuration...
Created PromptFoo configuration at: llm_test_results/promptfoo_config_20250106_213436.json

Configuration Details:
Provider Information:
Error creating config file: string indices must be integers, not 'str'
Failed to create configuration

Created test cases at: llm_test_results/test_cases_20250106_213436.json

Test Case Details:

Test 1:
Description: Basic Customer Service Test
Variables: {'customer_message': "My order #12345 hasn't arrived yet."}
Assertions: {'contains': 'order'}

Running evaluation...
Config file: llm_test_results/promptfoo_config_20250106_213436.json
Test cases: llm_test_results/test_cases_20250106_213436.json
Results will be saved to: llm_test_results/results_20250106_213436.json
PromptFoo execution failed:
Standard output: 
Standard error: (node:30807) [DEP0040] DeprecationWarning: The `punycode` module is deprecated. Ple